### Fill in if your running locally or in kaggle notebook, what model(s) you want to test and if you want to use the code for submission

In [1]:
Kaggle_Notebook=True #True #or False
SUBMIT_MODEL = "simple_prophet" #"simple_prophet"
SUBMISSION_CONFIG = True#False

# SUBMIT_MODEL
# Options ["lin_regr_base_norm", "lin_regr_base", "simple_prophet", "all_models"]
# If you choose "all_models", all models will be validated on the validation set also set SUBMISSION_CONFIG = False

# SUBMISSION_CONFIG
# If SUBMISSION_CONFIG == True
# - the actual prediction on the api is performed 
# - the validation set will not be tested to save time in the actual submission.

LEN_TRAIN=4357980 #Premeasured value.

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from prophet import Prophet
from datetime import datetime, timedelta
import sys
import pickle

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import logging

#logging.getLogger("prophet").setLevel(logging.WARNING)
#logging.getLogger("cmdstanpy").disabled = False
#logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/optiver-trading-at-the-close/public_timeseries_testing_util.py
/kaggle/input/optiver-trading-at-the-close/train.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/sample_submission.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/revealed_targets.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/test.csv
/kaggle/input/optiver-trading-at-the-close/optiver2023/competition.cpython-310-x86_64-linux-gnu.so
/kaggle/input/optiver-trading-at-the-close/optiver2023/__init__.py


### We need Python 3.10

In [3]:
!python --version

Python 3.10.12


In [4]:
import optiver2023
env = optiver2023.make_env()
iter_test = env.iter_test()

In [5]:
if SUBMIT_MODEL == "all_models" and SUBMISSION_CONFIG==True:
    print("If SUBMIT_MODEL == all_models, set SUBMISSION_CONFIG to False...")    
    SUBMISSION_CONFIG == False
    print(f"SUBMISSION_CONFIG is set to: {SUBMISSION_CONFIG}...")

In [6]:
if Kaggle_Notebook == True:    
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))
            
    INPUT_FILE_PATH="/kaggle/input/optiver-trading-at-the-close/"
    OUTPUT_FILE_PATH="/kaggle/working/"
else:
    INPUT_FILE_PATH="../data/"
    OUTPUT_FILE_PATH="../working/"

    if not os.path.exists(OUTPUT_FILE_PATH):
       os.makedirs(OUTPUT_FILE_PATH)
       print("The new directory is created!")
    else:
        print(f"{OUTPUT_FILE_PATH} Directory already exists!")

OUTPUT_FILE_PATH_MODELS = OUTPUT_FILE_PATH+'prophet_models.pkl'
CACHE_DIR=OUTPUT_FILE_PATH+'cache'
OUTPUT_FILE_PATH_MAPPED_DS = CACHE_DIR+'/dataset_mapped_0_400.pkl'

/kaggle/input/optiver-trading-at-the-close/public_timeseries_testing_util.py
/kaggle/input/optiver-trading-at-the-close/train.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/sample_submission.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/revealed_targets.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/test.csv
/kaggle/input/optiver-trading-at-the-close/optiver2023/competition.cpython-310-x86_64-linux-gnu.so
/kaggle/input/optiver-trading-at-the-close/optiver2023/__init__.py


In [7]:
if not os.path.exists(OUTPUT_FILE_PATH+'train.pkl'):    
    # low_memory = False
    usecols = ['target','date_id','row_id','stock_id']
    dtype_dict = {'target': 'float64', 'date_id': 'int', 'row_id': 'str', 'stock_id':'int'}
    data = pd.read_csv(INPUT_FILE_PATH+'train.csv', usecols = usecols, dtype=dtype_dict, low_memory = False)

    print(f"Creating training set...")
    train = data[data["date_id"] <= 400] 
    train.to_pickle(OUTPUT_FILE_PATH+'train.pkl')


    print(f"Creating validation set...")
    validation = data[data["date_id"] > 400] 
    validation.to_pickle(OUTPUT_FILE_PATH+'validation.pkl')
    
    
    print("Training and validation set were created")
else:
    print(f"Training and validation set already exists!")
    


Creating training set...
Creating validation set...
Training and validation set were created


In [8]:
if 'train' not in locals():
    print(f"Training and validation dataframe do not exists")
    train = pd.read_pickle(OUTPUT_FILE_PATH+'train.pkl') 
    validation = pd.read_pickle(OUTPUT_FILE_PATH+'validation.pkl')
else:
    print(f"Training and validation dataframe do exists")

Training and validation dataframe do exists


In [9]:
DATASETS = [train, validation]
MODELS = set() # use MODELS.add() to add items

### Prophet
#### Training data
For the simplest version, create one training set with columns ['timestamp', 'target'] <br/>
To Do maybe later, add more data to the model, like ['imbalance_buy_sell_flag', 'boolean for when far/near_price is']. <br/>
We already have a 'base' and 'base_norm' set we can use. <br/>

- Prophet can deal with NaN values, but we still have the option to use the (or create a new) y_target column that has the mean for missing values.
#### Model
For each stock id, fit the prohpet model and predict on the validate set <br/>

#### So we have 55 observations for 1 day
To use prophet we need to create a new column 'ds' that fits 55 observations per day. <br/>
We can do this by mapping each observation to a YYYY-MM-DD HH:MM:SS using ```create_datetime_mapping```


[Prophet non-daily_data](https://facebook.github.io/prophet/docs/non-daily_data.html)

In [10]:
#UNIQUE_STOCK_IDS=train['stock_id'].nunique()
#OBSERVATIONS_PER_DAY=len(pd.DataFrame(train[train['stock_id']==1][train['date_id']==0]))
#print(f"UNIQUE_STOCK_IDS: {UNIQUE_STOCK_IDS}\nOBSERVATIONS_PER_DAY: {OBSERVATIONS_PER_DAY}")

**disclaimer** <br/> 
The method ```create_datetime_mapping``` below is generated with Claude.ai, an LLM. Why: <br/> 
- Time saving
- Competition deadline has been expired over 2 years

In [11]:
def create_datetime_mapping(df, COLUMNS, save=False, row_id_col='row_id',date_id_col='date_id', cache_dir=CACHE_DIR):
    """
    Create datetime mapping using a pre-computed lookup table.
    This is the FASTEST method for large datasets.
    """
    
    # Create cache directory if it doesn't exist
    os.makedirs(cache_dir, exist_ok=True)
    
    # Determine cache filename based on row_id range
    row_id_min = df[row_id_col].iloc[0]
    row_id_max = df[row_id_col].iloc[-1]
    cache_filename = f"dataset_mapped_{row_id_min}_{row_id_max}.pkl"
    cache_path = os.path.join(cache_dir, cache_filename)
    
    # Check if cached version exists
    if os.path.exists(cache_path):
        print(f"Loading cached dataset from {cache_path}...")
        result_df = pd.read_pickle(cache_path)
        print("Cached dataset loaded successfully!")
        return result_df[COLUMNS]
    
    print("Configuring dataset for Prophet...")
    
    total_seconds_in_day = 24 * 60 * 60  # 86400
    values_per_id = 55
    interval_seconds = total_seconds_in_day / values_per_id
    
    # Create a copy
    df = df.reset_index()
    result_df = df.copy()
    
    # Extract time_index from row_id
    print("Extracting time_index from row_id...")
    result_df['time_index'] = result_df['row_id'].str.split('_').str[1].astype(int)
    
    # Create lookup table for all unique (date_id, time_index) combinations
    print("Building datetime lookup table...")
    unique_combinations = result_df[[date_id_col, 'time_index']].drop_duplicates()
    
    # Calculate timestamps for lookup table
    base_timestamp = pd.Timestamp('1970-01-01')
    unique_combinations['ds'] = (
        base_timestamp + 
        pd.to_timedelta(unique_combinations[date_id_col], unit='D') +
        pd.to_timedelta(unique_combinations['time_index'] * interval_seconds, unit='s')
    ).dt.strftime('%Y-%m-%d %H:%M:%S')
    
    # Merge lookup table back to main dataframe
    print("Merging lookup table...")
    result_df = result_df.merge(
        unique_combinations,
        on=[date_id_col, 'time_index'],
        how='left'
    )
    
    # Drop helper column
    result_df = result_df.drop('time_index', axis=1)
    
    # Save to cache
    if save:
        print(f"Saving to cache: {cache_path}")
        result_df[COLUMNS].to_pickle(cache_path)
        print("Saved to disk successfully!")
    
    return result_df[COLUMNS]

In [12]:
def fit_prophet_per_stockid(DATASET):
    prophet_models_stockid = {}
    
    DATASET = DATASET.rename(columns={"target": "y"}).drop(['row_id','index'], axis=1)
    
    for STOCK_ID in sorted(DATASET['stock_id'].unique()):
        if STOCK_ID%75 == 0:
            print(f"Fitting STOCK_ID: {STOCK_ID}...")
        # we only need yhat so uncertainty_samples=0, targets are "relative" so X_seasonality=False
        prophet_model  = Prophet(
            uncertainty_samples=0,
            changepoint_prior_scale=0.4, # it would be ineteresting to have a tuning here [0.01, 0.05, 0.1, 0.5]
            mcmc_samples=0, # it would be ineteresting to have a tuning here [0, 100, 500, 1000]
            yearly_seasonality=False,
            daily_seasonality=False,
            weekly_seasonality=False
        )
        DATASET_FIT = DATASET[DATASET['stock_id']==STOCK_ID]
        
        prophet_model.fit(DATASET_FIT)
        prophet_models_stockid['prophet_model_'+str(STOCK_ID)] = prophet_model
    
    print("Finished fitting models")
    
    # Store data (serialize)
    with open(OUTPUT_FILE_PATH_MODELS, 'wb') as handle:
        pickle.dump(prophet_models_stockid, handle)
    
    return prophet_models_stockid
    

### Hoping that the row_id is enough to validate against the test set, otherwise:
If this doesnt work  then maybe the index of the validation set should be remain the same and later fit back in the dataframe. <br/>

Meaning: dataset has index ```[0,1,2,3,4,5,6,7,8]``` <br/>
subset1 ```[0,3,6]```<br/>
subset2 ```[1,4,7]```<br/>
subset3 ```[2,5,8]```<br/><br/>

remerge back so that the postions match the original index. <br/>
yes: ```[0,1,2,3,4,5,6,7,8]``` <br/>
no: ```[0,3,6,1,4,7,2,5,8]``` <br/>

#### Edit
Did not take the risk for shenanigans, just added the index as column and later sort on this index.

In [13]:
def restruct_prophet_dataset(DATASET, DICTIONAIRY, MODEL):
    print(f"Reconstructing Dataset for model: {MODEL}... \n")
    prophet_predictions_df = pd.DataFrame()

    for df in DICTIONAIRY.values():
        prophet_predictions_df = pd.concat([prophet_predictions_df, pd.DataFrame(df) ], ignore_index=True)
    
    
    prophet_predictions_df = prophet_predictions_df.sort_values('index').set_index("index")
    
    #DATASET[MODEL+'_prediction'] = prophet_predictions_df['yhat']
   
    DATASET.loc[:, MODEL + '_prediction'] = DATASET['row_id'].map(prophet_predictions_df.set_index('row_id')['yhat'])
       
    MODELS.add(MODEL)
    
    print(f"Finished reconstructing Dataset for model: {MODEL}... \n")
    return DATASET #prophet_predictions_df


In [14]:
def predict_simple_prophet(DATASET, COLUMNS, MODEL):
    prophet_forecasts_stockid = {}    
    # This function does not have to run every time, just make one dataframe for one stock_id
    DATASET_MAPPED = create_datetime_mapping(DATASET, COLUMNS, save=True)
    
    print("Progress... \n")
    
    for STOCK_ID in sorted(DATASET_MAPPED['stock_id'].unique()):
        if STOCK_ID%75 == 0:
            print(f"Predict: STOCK_ID: {STOCK_ID} with model: {prophet_models_stockid.get('prophet_model_'+str(STOCK_ID))}")
        
        
        DATASET_STOCKID = DATASET_MAPPED[DATASET_MAPPED['stock_id']==STOCK_ID]
        DATASET_STOCKID_INDEX_ROW_ID = pd.DataFrame(DATASET_STOCKID[['index','row_id']]).reset_index(drop=True)
        DATASET_STOCKID_PREDICT = DATASET_STOCKID.drop(['index','row_id'], axis=1)

        #prophet_forecast = (prophet_models_stockid.get('prophet_model_'+str(STOCK_ID)).predict(DATASET_STOCKID_PREDICT)).drop('ds', axis=1)            
        #prophet_forecast[['row_id','index']] = DATASET_STOCKID_INDEX_ROW_ID[['row_id','index']]
        prophet_forecast = (
            prophet_models_stockid[f'prophet_model_{STOCK_ID}']
            .predict(DATASET_STOCKID_PREDICT)
            .drop(columns='ds')
            .join(DATASET_STOCKID_INDEX_ROW_ID[['row_id', 'index']])
        )

        prophet_forecasts_stockid['prophet_forecast_'+str(STOCK_ID)] = (prophet_forecast[['index','row_id','yhat']])
        
        if STOCK_ID == (len(DATASET['stock_id'].unique()) -1):
            print(f"Predict: STOCK_ID: {STOCK_ID} with model: {prophet_models_stockid.get('prophet_model_'+str(STOCK_ID))}")
    
    print(f"Finished....\n\n")
    
    df = restruct_prophet_dataset(DATASET, prophet_forecasts_stockid, MODEL)
    
    return df



## Fit the models
If Kaggle submission == True the validation set should not be tested, otherwise it should.

In [15]:
COLUMNS = ['index','target','row_id','stock_id','ds']
# Adjust dataframe
if not os.path.exists(OUTPUT_FILE_PATH_MAPPED_DS):
    print(f"{OUTPUT_FILE_PATH_MAPPED_DS} Does not exist!")
    train_simple_prophet = create_datetime_mapping(train, COLUMNS, save=True)
else:
    print(f"{OUTPUT_FILE_PATH_MAPPED_DS} exist!")
    # Load data (deserialize)
    with open(OUTPUT_FILE_PATH_MAPPED_DS, 'rb') as handle:
        train_simple_prophet = pickle.load(handle)

/kaggle/working/cache/dataset_mapped_0_400.pkl Does not exist!
Configuring dataset for Prophet...
Extracting time_index from row_id...
Building datetime lookup table...
Merging lookup table...
Saving to cache: /kaggle/working/cache/dataset_mapped_0_0_0_400_540_199.pkl
Saved to disk successfully!


In [16]:
# Fit on dataframe
if not os.path.exists(OUTPUT_FILE_PATH_MODELS):
    print(f"{OUTPUT_FILE_PATH_MODELS} does not exist!")
    prophet_models_stockid = fit_prophet_per_stockid(train_simple_prophet)
else:
    print(f"{OUTPUT_FILE_PATH_MODELS} exist!")
    # Load data (deserialize)
    with open(OUTPUT_FILE_PATH_MODELS, 'rb') as handle:
        prophet_models_stockid = pickle.load(handle)

/kaggle/working/prophet_models.pkl does not exist!
Fitting STOCK_ID: 0...


11:00:04 - cmdstanpy - INFO - Chain [1] start processing
11:00:09 - cmdstanpy - INFO - Chain [1] done processing
11:00:09 - cmdstanpy - INFO - Chain [1] start processing
11:00:15 - cmdstanpy - INFO - Chain [1] done processing
11:00:15 - cmdstanpy - INFO - Chain [1] start processing
11:00:20 - cmdstanpy - INFO - Chain [1] done processing
11:00:21 - cmdstanpy - INFO - Chain [1] start processing
11:00:28 - cmdstanpy - INFO - Chain [1] done processing
11:00:28 - cmdstanpy - INFO - Chain [1] start processing
11:00:35 - cmdstanpy - INFO - Chain [1] done processing
11:00:35 - cmdstanpy - INFO - Chain [1] start processing
11:00:42 - cmdstanpy - INFO - Chain [1] done processing
11:00:42 - cmdstanpy - INFO - Chain [1] start processing
11:00:49 - cmdstanpy - INFO - Chain [1] done processing
11:00:50 - cmdstanpy - INFO - Chain [1] start processing
11:00:54 - cmdstanpy - INFO - Chain [1] done processing
11:00:54 - cmdstanpy - INFO - Chain [1] start processing
11:00:59 - cmdstanpy - INFO - Chain [1]

Fitting STOCK_ID: 75...


11:08:34 - cmdstanpy - INFO - Chain [1] start processing
11:08:43 - cmdstanpy - INFO - Chain [1] done processing
11:08:43 - cmdstanpy - INFO - Chain [1] start processing
11:08:49 - cmdstanpy - INFO - Chain [1] done processing
11:08:50 - cmdstanpy - INFO - Chain [1] start processing
11:08:55 - cmdstanpy - INFO - Chain [1] done processing
11:08:55 - cmdstanpy - INFO - Chain [1] start processing
11:09:02 - cmdstanpy - INFO - Chain [1] done processing
11:09:02 - cmdstanpy - INFO - Chain [1] start processing
11:09:07 - cmdstanpy - INFO - Chain [1] done processing
11:09:07 - cmdstanpy - INFO - Chain [1] start processing
11:09:13 - cmdstanpy - INFO - Chain [1] done processing
11:09:13 - cmdstanpy - INFO - Chain [1] start processing
11:09:18 - cmdstanpy - INFO - Chain [1] done processing
11:09:18 - cmdstanpy - INFO - Chain [1] start processing
11:09:24 - cmdstanpy - INFO - Chain [1] done processing
11:09:24 - cmdstanpy - INFO - Chain [1] start processing
11:09:31 - cmdstanpy - INFO - Chain [1]

Fitting STOCK_ID: 150...


11:17:10 - cmdstanpy - INFO - Chain [1] start processing
11:17:14 - cmdstanpy - INFO - Chain [1] done processing
11:17:14 - cmdstanpy - INFO - Chain [1] start processing
11:17:22 - cmdstanpy - INFO - Chain [1] done processing
11:17:23 - cmdstanpy - INFO - Chain [1] start processing
11:17:27 - cmdstanpy - INFO - Chain [1] done processing
11:17:27 - cmdstanpy - INFO - Chain [1] start processing
11:17:32 - cmdstanpy - INFO - Chain [1] done processing
11:17:32 - cmdstanpy - INFO - Chain [1] start processing
11:17:44 - cmdstanpy - INFO - Chain [1] done processing
11:17:45 - cmdstanpy - INFO - Chain [1] start processing
11:17:52 - cmdstanpy - INFO - Chain [1] done processing
11:17:52 - cmdstanpy - INFO - Chain [1] start processing
11:17:56 - cmdstanpy - INFO - Chain [1] done processing
11:17:56 - cmdstanpy - INFO - Chain [1] start processing
11:18:03 - cmdstanpy - INFO - Chain [1] done processing
11:18:03 - cmdstanpy - INFO - Chain [1] start processing
11:18:09 - cmdstanpy - INFO - Chain [1]

Finished fitting models


### Evaluation on validation set
Our evaluation metric is Mean Absolute Error (MAE). The formula is given by:

MAE=1nn∑i=1 |yi−xi| 
Where: n is the total number of data points.
yi  is the predicted value for the  ith  data point.
xi  is the observed value for the  ith  data point.

In [17]:
if not SUBMISSION_CONFIG:
    MAE_DICT = {}
    def evaluation(MODELS):
        for MODEL in MODELS:
            mae = (validation[MODEL+'_prediction'] - validation['target']).abs().mean()
            MAE_DICT.update({MODEL : mae})
            
    val_forecast_prophet = predict_simple_prophet(validation, COLUMNS, "simple_prophet")
    evaluation(MODELS)

    SORTED_MAE_DICT=dict(sorted(MAE_DICT.items(), key=lambda item: item[1])) 

    for key, value in SORTED_MAE_DICT.items():
        print(f"{key}: {value}")

#### Randomly remove lines from validation set, script should work.

In [18]:
if not SUBMISSION_CONFIG:
    def evaluation_reduced(MODEL_RED):  
        mae = (validation_reduced[MODEL_RED+'_prediction'] - validation_reduced['target']).abs().mean()
        MAE_DICT_RED.update({MODEL_RED : mae})
    
    MODEL_RED = "simple_prophet_reduced"
    MAE_DICT_RED = {}
    
    validation_reduced = validation.drop(validation.sample(frac=0.2, random_state=42).index)
    validation_reduced = validation_reduced[validation_reduced["stock_id"] != 4]
    validation_reduced = validation_reduced[validation_reduced["date_id"] != 405]
    
    val_reduced_forecast_prophet = predict_simple_prophet(validation_reduced, COLUMNS, MODEL_RED)
    evaluation_reduced(MODEL_RED)

    SORTED_MAE_RED_DICT=dict(sorted(MAE_DICT_RED.items(), key=lambda item: item[1])) 

    for key, value in SORTED_MAE_RED_DICT.items():
        print(f"{key}: {value}")

if Kaggle_Notebook == False:
    fixed_columns = ['row_id','time_id']
    def save_predictions_to_file(MODELS):    
        print("Saving models to file...\n")
        for MODEL in MODELS:
            PREDICTION_COLUMN = MODEL+'_prediction'
            columns = [MODEL+'_prediction']
            columns.extend(fixed_columns)
            columns.reverse()
            #print(f"columns: {columns}")
            file_name = OUTPUT_FILE_PATH+'/submissions/'+PREDICTION_COLUMN+'.csv'
            
            #print(test[columns].rename(columns={PREDICTION_COLUMN:'target'}, inplace=True).head())
            test[columns].rename(columns={PREDICTION_COLUMN:"target"}).to_csv(file_name, index=False, header=True)        
            print(f"Model: {MODEL} to file: {file_name}")
        print("\n\nFinished saving models")
            
    save_predictions_to_file(MODELS)  

# ----------------------Submit the models----------------

In [19]:
def submit(MODEL):    
    print(f"Model: {MODEL} selected for submission...")
    
    PREDICTION_COLUMN = MODEL+'_prediction'
    
        
    for (test, revealed_targets, sample_prediction) in iter_test:
        
        print(f"Submitting: {MODEL}...")        
        COLUMNS = ['index','row_id','stock_id','ds']
        predictions = predict_simple_prophet(test, COLUMNS, MODEL)
        
        #sample_prediction.loc[:, 'target'] = sample_prediction['row_id'].map(predictions.set_index('row_id')[PREDICTION_COLUMN])
        sample_prediction['target'] = sample_prediction['row_id'].map(predictions.set_index('row_id')[PREDICTION_COLUMN])

        env.predict(sample_prediction)

    
    print("\n\nFinished submitting models")

In [20]:
# Make submission if true
if SUBMISSION_CONFIG:
    MODEL = SUBMIT_MODEL
    submit(MODEL)

Model: simple_prophet selected for submission...
This version of the API is not optimized and should not be used to estimate the runtime of your code on the hidden test set.
Submitting: simple_prophet...
Configuring dataset for Prophet...
Extracting time_index from row_id...
Building datetime lookup table...
Merging lookup table...
Saving to cache: /kaggle/working/cache/dataset_mapped_478_0_0_478_0_199.pkl
Saved to disk successfully!
Progress... 

Predict: STOCK_ID: 0 with model: <prophet.forecaster.Prophet object at 0x7b481aaa8550>
Predict: STOCK_ID: 75 with model: <prophet.forecaster.Prophet object at 0x7b47c4dbdab0>
Predict: STOCK_ID: 150 with model: <prophet.forecaster.Prophet object at 0x7b47b0bcf640>
Predict: STOCK_ID: 199 with model: <prophet.forecaster.Prophet object at 0x7b47a55ff8b0>
Finished....


Reconstructing Dataset for model: simple_prophet... 

Finished reconstructing Dataset for model: simple_prophet... 

predictions Head:    stock_id  date_id  seconds_in_bucket  imba